# Act 1 — Keys and secrets

Notebook 02 covered who is allowed to do what; this notebook covers how the data itself is protected and how the workloads access it without leaking credentials. The first act is the foundation: where keys live, where secrets live, and which services to use to manage both.

Three products: **Cloud KMS** for encryption keys, **Secret Manager** for application secrets, and **Cloud HSM** as the FIPS-grade backend for KMS when you need it.

## Cloud KMS

**Cloud KMS** stores and operates encryption keys. The structure:

- **Project** contains **key rings** (logical groupings, location-scoped).
- **Key ring** contains **crypto keys** (each key has a purpose: symmetric encrypt/decrypt, asymmetric encrypt, asymmetric sign, MAC).
- **Crypto key** contains **key versions** (the actual material; rotated by creating new versions; old versions can decrypt/verify but new operations use the primary version).

**Locations matter for KMS.** A key is regional, multi-region, or global. When you use CMEK on a resource, the key must be in a compatible location — a GCS bucket in `us-central1` needs a key in `us-central1` or a compatible multi-region.

**Rotation is automatic** when you set a rotation period; new versions are minted on schedule. Old versions stay in `enabled` state for decrypting old data. Destroying a key version is reversible for 24 hours, then permanent — that's the kill switch for revoking data access.

**Key purposes:**

- **Symmetric encrypt/decrypt** (default for CMEK on GCS, BigQuery, Cloud SQL, Persistent Disk).
- **Asymmetric encrypt / sign / MAC** for application-level crypto.
- **External Key Manager (EKM)** — key material lives in a third-party HSM (or another cloud), KMS proxies operations to it. Used when regulation forbids any GCP-resident key material.

**Cloud HSM** is a backend choice for any crypto key: the underlying key is FIPS 140-2 Level 3 hardware-backed instead of software-backed. Same API; higher cost per operation.

## Secret Manager

**Secret Manager** stores application secrets — DB passwords, API keys, OAuth client secrets, certificates. Structure:

- **Secret** — a named container.
- **Secret version** — the actual value. Versions are immutable; you add new versions when the secret changes.
- Access via API/SDK using IAM (`roles/secretmanager.secretAccessor`).

**Two replication policies:**

- **Automatic** — Google chooses replication regions. Highest availability.
- **User-managed** — you pick the regions. Used for residency requirements.

**Automatic rotation** hooks let you call a Cloud Function or webhook on a schedule (the function does the actual credential rotation against the upstream and writes the new value as a new version).

**Patterns:**

- **At-runtime fetch** — application calls Secret Manager on startup or per-request, caches the value. Pair with Workload Identity so no SA key is involved.
- **Cloud Run secret env vars** — Cloud Run can resolve a secret into an env var at deploy time. Easy for simple cases; the value lives in the Cloud Run revision configuration (not in your source).
- **Cloud Run secret volumes** — mount the secret as a file. Used when an application reads from disk (e.g. TLS certificates).

**Compare:** Secret Manager ≈ AWS Secrets Manager ≈ Azure Key Vault for secrets. KMS is the key-material layer; Secret Manager is the application-value layer. Don't conflate them — store credentials in Secret Manager, encryption keys in KMS.

# Act 2 — Workload and human access without VPN

GCP has a coherent answer to the "how do I let employees and workloads reach private resources securely" question, and that answer is mostly **not** a VPN. Identity-Aware Proxy handles human access; Workload Identity (recap from notebook 02) handles workload access.

## Identity-Aware Proxy (IAP) — zero-trust web access

**IAP** sits in front of web apps and TCP services. It authenticates the caller (Google identity, optionally with Conditional Access via Access Context Manager), checks IAM on the resource (`roles/iap.httpsResourceAccessor` for web, `roles/iap.tunnelResourceAccessor` for TCP), and forwards the request to your backend.

**Two modes:**

- **IAP for HTTPS** — protects apps behind a Global External Application LB (web apps, internal tools, dev portals). User browses to the URL → Google sign-in → IAP checks IAM → traffic flows to your Cloud Run / GKE / GCE backend.
- **IAP for TCP** — proxies SSH (`gcloud compute ssh --tunnel-through-iap`), RDP, and arbitrary TCP to private VMs. Replaces bastion hosts.

**Why it matters.** Before IAP, the standard pattern was "VPN into the VPC, then talk to internal services." With IAP, internal services are exposed *over the internet* but gated by Google identity + IAM — no VPN, no bastion. This is BeyondCorp / zero-trust as a configuration.

IAP can layer **Access Context Manager** levels on top — "only allow from corporate IPs *and* trusted devices *and* office hours" — for sensitive resources. Add `Cloud Identity-Aware Proxy + Context-Aware Access` for the full zero-trust shape.

## Workload Identity recap

Covered in notebook 02; recap because it belongs in the security toolbelt:

- **GKE Workload Identity** — Kubernetes SA impersonates a Google SA. No mounted JSON keys.
- **Workload Identity Federation** — external workloads (GitHub Actions, AWS IAM roles) impersonate a Google SA via the two-step STS + generateAccessToken dance.

The security goal both achieve: **no long-lived service account JSON keys exist anywhere.** Org Policy `iam.disableServiceAccountKeyCreation` enforces this. The combination is GCP's most-distinctive security primitive — adopt it everywhere.

# Act 3 — Workload integrity

Knowing who can talk to your services is half the security story. The other half is knowing *what code is running* in those services. Two products cover this: **Binary Authorization** for container image provenance, and **Confidential Computing** for memory-level isolation.

## Binary Authorization

**Binary Authorization** enforces a policy at deploy time: only container images that meet your rules can run on GKE, Cloud Run, or Cloud Run for Anthos.

**The model:**

- **Attestors** — signers (Cloud KMS keys) that vouch for an image ("this image passed our vulnerability scan," "this image was built by the prod Cloud Build pipeline").
- **Policy** — "deployment requires attestations from attestors X and Y." Per-project, per-cluster, per-namespace granularity.
- **Enforcement modes** — `enforced` blocks non-compliant deploys, `dryrun` logs without blocking.

**Typical chain:**

1. Cloud Build runs the build, scans the image with Container Analysis.
2. If the scan passes, Cloud Build calls an attestor (a Cloud KMS key) to sign the image digest.
3. Cloud Deploy / kubectl deploys the image.
4. Binary Authorization checks the policy: "is there an attestation from our `vuln-scan-passed` attestor and our `prod-pipeline` attestor?" Yes → run. No → reject.

This is the right answer to "how do I prove only reviewed, scanned, and built-by-the-right-pipeline code runs in prod?"

## Confidential Computing

**Confidential VMs** and **Confidential GKE Nodes** use AMD SEV (or Intel TDX on newer instances) to encrypt VM memory with a key the hypervisor doesn't have. The result: even GCP's own infrastructure can't read your process memory.

Use case is narrow: extremely sensitive workloads where you need cryptographic proof that no party (including the hypervisor) can read your memory. Common in defence, healthcare, multi-party computation. For most workloads, the standard hypervisor isolation is enough — Confidential Computing is the answer to a specific threat model, not a general default.

# Act 4 — Detection, posture, and edge protection

The last act is the security operations surface — what you turn on to actually *see* what's happening across the org. **Security Command Center** is the umbrella; **Cloud Armor** (recap from notebook 07) is the edge; **reCAPTCHA Enterprise** is the bot-protection layer.

## Security Command Center

**Security Command Center (SCC)** is GCP's central security-posture and threat-detection platform. Three tiers:

- **Standard** — free. Some posture findings, IAM Recommender, basic Web Security Scanner. The minimum baseline.
- **Premium** — paid. Full posture management, Event Threat Detection (anomalous logins, IAM grants, …), Container Threat Detection, Continuous Exports to Pub/Sub or BigQuery.
- **Enterprise** — adds case management, SOAR-style playbooks, more integrations. The full SOC product.

**Findings come from many sources:**

- **Cloud DLP / Sensitive Data Protection** — finds PII in your data.
- **Container Analysis** — vulnerability scan results from Artifact Registry.
- **Event Threat Detection** — anomalies in audit logs.
- **Web Security Scanner** — web-app vulnerabilities.
- **Custom sources** — third-party tools push findings via the API.

**Posture management** evaluates your configuration against benchmarks (CIS, NIST) and surfaces gaps — "this bucket is public," "this VM has a public IP without a firewall rule," "this SA has owner permissions." Continuous Exports send findings to BigQuery for analysis or Pub/Sub for downstream automation.

SCC Premium is the right baseline for any organisation with more than a handful of projects.

## Cloud Armor — recap from notebook 07

The edge WAF. Two policy types worth re-emphasising:

- **Security policy** — attached to a backend service. Evaluates at the GFE, before the backend.
- **Edge security policy** — applied earlier, before Cloud CDN cache decisions.

For a deeply layered defence: Edge security policy → Security policy → IAP → Backend IAM. Each layer is independent and additive.

## reCAPTCHA Enterprise

**reCAPTCHA Enterprise** is the production version of the well-known reCAPTCHA bot-detection. It gives you risk scores per request (0.0 = bot, 1.0 = human) rather than a binary pass/fail. Common shapes:

- **Score-based** — reCAPTCHA runs invisibly; your app uses the score to decide friction (allow, MFA, block).
- **Account defender** — detect compromised accounts based on behaviour patterns.
- **MFA via WebAuthn** — bundled MFA backed by reCAPTCHA risk signals.

**Pairs with Cloud Armor** — reCAPTCHA scores can drive Cloud Armor rules at the edge.

## What carries into later chapters

KMS and Secret Manager are quietly everywhere — CMEK on storage (notebook 05), secrets in Cloud Run env vars (notebook 04), encrypted Cloud SQL backups (notebook 08). IAP is the right answer for any internal-tool admin UI in notebook 13's CI/CD work. SCC findings feed into the governance posture in notebook 12.

Three habits to carry forward:

- **No SA JSON keys, anywhere.** WIF for external workloads, Workload Identity for GKE, attached SAs for everything else. Enforce via Org Policy.
- **IAP over VPN for human access.** Less infrastructure, finer-grained, MFA built in.
- **SCC Premium turned on early.** Cheaper than the first incident it catches.